# PII Redaction for Revision Chain (RC) Logs

## Description

RC logs may contain personal information that needs to be protected before analysis, storage, or sharing. This notebook sanitizes RC logs by identifying and redacting personally identifiable information (PII) while preserving the original structure.

**Output:** A structure-preserving, redacted JSON file that can be safely used for analysis and sharing.

In [ ]:
# Import Required Libraries
import json
import re
from copy import deepcopy

# No external libraries or APIs used

In [ ]:
# Define PII Patterns

# Email pattern
EMAIL_REGEX = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

# Phone numbers (supports international format, parentheses, spaces, hyphens)
PHONE_REGEX = r'(\+?\d{1,3}[\s-]?)?\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{4}'

# Student ID (7-10 digits)
STUDENT_ID_REGEX = r'\b\d{7,10}\b'

# Name heuristic (looks for "my name is" or "i am" patterns)
NAME_REGEX = r'(my name is|i am)\s+[A-Z][a-z]+(\s+[A-Z][a-z]+)?'

In [ ]:
# Implement Text Redaction Function
def redact_text(text: str) -> str:
    """
    Redacts PII from text using regex patterns.
    
    Args:
        text: Input string to redact
        
    Returns:
        String with PII replaced by placeholder text
    """
    if not isinstance(text, str):
        return text

    # Redact email addresses
    text = re.sub(EMAIL_REGEX, "[REDACTED_EMAIL]", text)
    
    # Redact phone numbers
    text = re.sub(PHONE_REGEX, "[REDACTED_PHONE]", text)
    
    # Redact student IDs
    text = re.sub(STUDENT_ID_REGEX, "[REDACTED_STUDENT_ID]", text)
    
    # Redact names (case-insensitive)
    text = re.sub(NAME_REGEX, "[REDACTED_NAME]", text, flags=re.IGNORECASE)

    return text

In [ ]:
# Implement RC Log Redaction Function
def redact_rc_log(rc_log: dict) -> dict:
    """
    Redacts PII from RC log while preserving structure.
    
    Args:
        rc_log: Dictionary containing RC log data
        
    Returns:
        Redacted RC log dictionary with same structure
    """
    redacted_log = deepcopy(rc_log)

    if "revision_chain" in redacted_log:
        for step in redacted_log["revision_chain"]:
            # Redact student prompts
            if "student_prompt" in step:
                step["student_prompt"] = redact_text(step["student_prompt"])
            
            # Redact LLM responses
            if "llm_response" in step:
                step["llm_response"] = redact_text(step["llm_response"])

    return redacted_log

In [ ]:
# Load Sample RC Log
sample_rc_log = {
    "rc_log_id": "rc_test_001",
    "rubric_id": "rub_it_0023",
    "revision_chain": [
        {
            "step_index": 0,
            "student_prompt": "Hi, my name is John Smith. My student ID is 223553052 and my email is john.smith@university.edu. My phone is +1-555-123-4567.",
            "llm_response": "Thanks John, I can help you. I've noted your student ID 223553052 and will contact you at john.smith@university.edu or call 5551234567 if needed."
        },
        {
            "step_index": 1,
            "student_prompt": "I am Jane Doe and my ID is 987654321. Please send updates to jane.doe@college.edu.",
            "llm_response": "Hello Jane! I've recorded your information for student 987654321 and will use jane.doe@college.edu for communications."
        }
    ]
}

In [ ]:
# Demonstrate Before vs After
print("=== ORIGINAL RC LOG ===")
print(json.dumps(sample_rc_log, indent=2))

redacted = redact_rc_log(sample_rc_log)

print("\n=== REDACTED RC LOG ===")
print(json.dumps(redacted, indent=2))

In [ ]:
# Save Redacted Output
output_filename = "rc_test_001_redacted.json"

with open(output_filename, "w") as f:
    json.dump(redacted, f, indent=2)

print(f"Redacted RC log saved to: {output_filename}")
print(f"File size: {len(json.dumps(redacted, indent=2))} characters")

## Final Notes

### PII Types Handled:
- **Email addresses**: All standard email formats
- **Phone numbers**: Including international formats, parentheses, spaces, hyphens
- **Student IDs**: 7-10 digit sequences
- **Names**: Simple heuristic for "my name is" and "i am" patterns

### Limitations (What is NOT handled):
- **Advanced NLP cases**: Names mentioned without explicit patterns
- **Contextual PII**: Addresses, dates of birth, etc.
- **Obfuscated PII**: Partial or encoded personal information
- **Semantic analysis**: Understanding context to identify PII

### Purpose:
This notebook provides a **baseline privacy layer** for RC logs. It should be used as part of a comprehensive data protection strategy, not as a complete solution for all privacy requirements.

### Recommendations:
1. Review redacted outputs before sharing
2. Consider additional privacy measures for sensitive use cases
3. Update regex patterns based on your specific data characteristics
4. Test with representative samples of your RC logs